# ChE629 Course Project

In [1]:
import torch
import numpy as np
import pandas as pd
import sklearn
import os
import pickle
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
# try:
#     from rdkit import Chem
#     from rdkit.Chem import Descriptors, AllChem
#     print("RDKit already installed")
# except ImportError:
#     import subprocess
#     import sys
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "rdkit"])
#     from rdkit import Chem
#     from rdkit.Chem import Descriptors, AllChem
#     print("RDKit installed and imported successfully")

In [3]:
# try:
#     from Bio import SeqIO
#     print("Biopython already installed")
# except ImportError:
#     import subprocess
#     import sys
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "biopython"])
#     from Bio import SeqIO
#     print("Biopython installed and imported successfully")

In [4]:
print("All libraries working perfectly")

All libraries working perfectly


In [5]:
# Local paths
root = "/kaggle/input/datasets/nonitgupta/drug-target-interactions/"

davis_path = os.path.join(root, "EDTA-main/data/davis/SMILES_affinity.csv")
kiba_path  = os.path.join(root, "EDTA-main/data/kiba/SMILES_affinity.csv")

# Load datasets
davis_data = pd.read_csv(davis_path)
kiba_data = pd.read_csv(kiba_path)

print("DAVIS shape:", davis_data.shape)
print("KIBA shape:", kiba_data.shape)

DAVIS shape: (30056, 3)
KIBA shape: (118254, 3)


#### Input Features Processing

b) Drug Features

In [6]:
max_atoms = 50
feature_dim = 24

print(f'Maximum atoms for features are taken into account (per molecule): {max_atoms}')

def pad_or_truncate(mat):
    num_atoms = mat.shape[0]

    # truncate
    if num_atoms > max_atoms:
        return mat[:max_atoms, :]

    # pad
    if num_atoms < max_atoms:
        pad_rows = max_atoms - num_atoms
        padding = np.zeros((pad_rows, feature_dim))
        return np.vstack([mat, padding])

    return mat


def load_drug_tensor(pkl_path):

    daf = pd.read_pickle(pkl_path)

    # pad/truncate all molecules
    daf_fixed = {k: pad_or_truncate(v) for k, v in daf.items()}

    # stack matrices
    matrix_stack = np.stack(list(daf_fixed.values()))

    # convert to tensor
    drug_tensor = torch.tensor(matrix_stack, dtype=torch.float32)

    return drug_tensor, list(daf_fixed.keys())


# ===============================
# KIBA
# ===============================

kiba_path  = os.path.join(root, "EDTA-main/data/kiba/Drug_Atomic_Feature24.pkl")

kiba_drug_tensor, kiba_drug_keys = load_drug_tensor(kiba_path)

print("KIBA tensor shape:", kiba_drug_tensor.shape)


# ===============================
# DAVIS
# ===============================

davis_path = os.path.join(root, "EDTA-main/data/davis/Drug_Atomic_Feature24.pkl")

davis_drug_tensor, davis_drug_keys = load_drug_tensor(davis_path)

print("DAVIS tensor shape:", davis_drug_tensor.shape)

Maximum atoms for features are taken into account (per molecule): 50
KIBA tensor shape: torch.Size([2111, 50, 24])
DAVIS tensor shape: torch.Size([68, 50, 24])


a) Drug SMILES

This character level enconding could be replaced with atom level encoding. Like convert C,l to Cl and B,r to Br respectively.

In [7]:
# Combine SMILES from both datasets
combined_smiles = pd.concat([davis_data['ligand'], kiba_data['ligand']])

# Build vocabulary
chars = sorted(set("".join(combined_smiles)))
char_to_int = {c: i+1 for i, c in enumerate(chars)}

print("Vocabulary size:", len(chars))
print("Characters:", chars)
SMILES_VOCAB_SIZE = len(chars) + 1

Vocabulary size: 32
Characters: ['#', '(', ')', '+', '-', '.', '/', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=', '@', 'B', 'C', 'F', 'H', 'I', 'N', 'O', 'P', 'S', '[', '\\', ']', 'l', 'r']


In [8]:
unknown_chars = set()
for smiles in kiba_drug_keys:
    for c in smiles:
        if c not in char_to_int:
            unknown_chars.add(c)
print("Unknown characters in KIBA drug keys:", unknown_chars)  # should be empty set

Unknown characters in KIBA drug keys: set()


In [9]:
unknown_chars = set()
for smiles in davis_drug_keys:
    for c in smiles:
        if c not in char_to_int:
            unknown_chars.add(c)
print("Unknown characters in DAVIS drug keys:", unknown_chars)  # should be empty set

Unknown characters in DAVIS drug keys: set()


In [10]:
# Encoding function
def encode_smiles(smiles):
    return [char_to_int[c] for c in smiles]

# Set max length
max_len = 100
print("Using max SMILES length:", max_len)

# Pad / truncate
def pad_or_truncate_smiles(vec):
    if len(vec) > max_len:
        return vec[:max_len]
    return vec + [0]*(max_len - len(vec))

# ---- DAVIS ----
# Build SMILES tensor indexed by davis_drug_keys (same order as drug_atomic_tensor)
davis_vectors = [
    pad_or_truncate_smiles(encode_smiles(smiles)) 
    for smiles in davis_drug_keys  # iterate over pkl keys, not the full CSV
]
davis_smiles_tensor = torch.tensor(np.stack(davis_vectors), dtype=torch.long)
print("Davis tensor shape:", davis_smiles_tensor.shape)

# ---- KIBA ----
# Build SMILES tensor in the same order as kiba_drug_keys (from pkl)
kiba_vectors = [
    pad_or_truncate_smiles(encode_smiles(smiles))
    for smiles in kiba_drug_keys   # 2111 unique drugs, same order as drug_atomic_tensor
]
kiba_smiles_tensor = torch.tensor(np.stack(kiba_vectors), dtype=torch.long)
print("KIBA SMILES tensor shape:", kiba_smiles_tensor.shape)   # (2111, 100)

Using max SMILES length: 100
Davis tensor shape: torch.Size([68, 100])
KIBA SMILES tensor shape: torch.Size([2111, 100])


c) Target Feature + BPE

In [11]:
# ── Load merge rules ──────────────────────────────────────
merge_rules_path = os.path.join(root, "EDTA-main/data/davis/protein_codes_uniprot.txt")

merge_rules = []
with open(merge_rules_path, "r") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.strip().split()
        if len(parts) == 2:
            merge_rules.append((parts[0], parts[1]))

print(f"Total merge rules: {len(merge_rules)}")

# ── Load vocabulary ───────────────────────────────────────
vocab_path = os.path.join(root, "EDTA-main/data/davis/subword_units_map_uniprot.csv")
vocab_df = pd.read_csv(vocab_path)
print(f"Total vocab size: {len(vocab_df)}")

token_to_id = dict(zip(vocab_df['index'], vocab_df['level_0']))

# ── Precompute merge rules as a rank dict for O(1) pair lookup ───────────────
merge_rank = {(a, b): i for i, (a, b) in enumerate(merge_rules)}

# ── Padding helper ────────────────────────────────────────────────────────────
def pad_target_feature(x, max_len):
    x = np.asarray(x)
    L, d = x.shape
    if L >= max_len:
        return x[:max_len]
    padded = np.zeros((max_len, d))
    padded[:L] = x
    return padded

# ── Fast BPE encoding function ────────────────────────────────────────────────
def apply_bpe_fast(sequence, merge_rank, token_to_id, max_len, unk_id=0):
    tokens = list(sequence)

    while True:
        best_rank = float("inf")
        best_idx  = -1
        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            rank = merge_rank.get(pair, float("inf"))
            if rank < best_rank:
                best_rank = rank
                best_idx  = i

        if best_idx == -1:
            break

        merged = tokens[best_idx] + tokens[best_idx + 1]
        pair_a = tokens[best_idx]
        pair_b = tokens[best_idx + 1]
        new_tokens = []
        i = 0
        while i < len(tokens):
            if (i < len(tokens) - 1
                    and tokens[i]     == pair_a
                    and tokens[i + 1] == pair_b):
                new_tokens.append(merged)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

    ids = [token_to_id.get(t, unk_id) + 1 for t in tokens]

    if len(ids) >= max_len:
        return ids[:max_len]
    return ids + [0] * (max_len - len(ids))

# ── Worker function for parallel file loading ─────────────────────────────────
def _load_one(args):
    file, folder_path, bpe_max_len, merge_rank, token_to_id = args
    protein_id = file.replace(".pkl", "")

    with open(os.path.join(folder_path, file), "rb") as f:
        data = pickle.load(f)

    seq_feat = pad_target_feature(data["seq_feat"], MAX_TARGET_LEN)
    bpe_ids  = apply_bpe_fast(str(data["seq"]), merge_rank, token_to_id, bpe_max_len)

    return protein_id, seq_feat, bpe_ids

# ── Constants ─────────────────────────────────────────────────────────────────
MAX_TARGET_LEN    = 500
MAX_BPE_LEN_DAVIS = 400
MAX_BPE_LEN_KIBA  = 556

# ── Updated target feature loader (parallelised) ──────────────────────────────
def load_target_features_fast(folder_path, bpe_max_len, max_workers=8):

    files = sorted([f for f in os.listdir(folder_path) if f.endswith(".pkl")])

    protein_ids     = [None] * len(files)
    target_features = [None] * len(files)
    target_bpe      = [None] * len(files)

    args_list = [
        (file, folder_path, bpe_max_len, merge_rank, token_to_id)
        for file in files
    ]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_load_one, a): i for i, a in enumerate(args_list)}
        for future in as_completed(futures):
            i = futures[future]
            protein_ids[i], target_features[i], target_bpe[i] = future.result()

    target_features = torch.tensor(np.stack(target_features), dtype=torch.float32)
    target_bpe      = torch.tensor(np.array(target_bpe),      dtype=torch.long)

    return target_features, target_bpe, protein_ids

# ── Load Davis and KIBA ───────────────────────────────────────────────────────
print("Loading DAVIS...")
davis_feat, davis_bpe, davis_ids = load_target_features_fast(
    os.path.join(root, "EDTA-main/data/davis/Target_Feature+BPE"),
    MAX_BPE_LEN_DAVIS
)
print(f"davis_feat : {davis_feat.shape}  dtype={davis_feat.dtype}")
print(f"davis_bpe  : {davis_bpe.shape}   dtype={davis_bpe.dtype}")

print("\nLoading KIBA...")
kiba_feat, kiba_bpe, kiba_ids = load_target_features_fast(
    os.path.join(root, "EDTA-main/data/kiba/Target_Feature+BPE"),
    MAX_BPE_LEN_KIBA
)
print(f"kiba_feat : {kiba_feat.shape}  dtype={kiba_feat.dtype}")
print(f"kiba_bpe  : {kiba_bpe.shape}   dtype={kiba_bpe.dtype}")

Total merge rules: 10120
Total vocab size: 16693
Loading DAVIS...
davis_feat : torch.Size([442, 500, 42])  dtype=torch.float32
davis_bpe  : torch.Size([442, 400])   dtype=torch.int64

Loading KIBA...
kiba_feat : torch.Size([228, 500, 42])  dtype=torch.float32
kiba_bpe  : torch.Size([228, 556])   dtype=torch.int64


d) Potential Residue

In [12]:
# =====================================================
# PATHS
# =====================================================
davis_residue_path = os.path.join(root, "EDTA-main/data/davis/Potential_Residue_Feature")
kiba_residue_path  = os.path.join(root, "EDTA-main/data/kiba/Potential_Residue_Feature")

# =====================================================
# CONSTANTS
# =====================================================
MAX_RESIDUE_LEN = 200
FEATURE_DIM = 67

# =====================================================
# PADDING FUNCTION
# =====================================================

def pad_residue_feature(mat):
    mat = np.asarray(mat)
    L, d = mat.shape

    # truncate
    if L >= MAX_RESIDUE_LEN:
        return mat[:MAX_RESIDUE_LEN]

    # pad
    padded = np.zeros((MAX_RESIDUE_LEN, d))
    padded[:L] = mat
    return padded

# =====================================================
# LOADER FUNCTION
# =====================================================

def load_residue_tensor(folder_path):
    residue_features = []
    protein_ids = []

    files = sorted(os.listdir(folder_path))
    for file in files:
        if not file.endswith(".pkl"):
            continue

        protein_id = file.replace(".pkl", "")
        protein_ids.append(protein_id)
        path = os.path.join(folder_path, file)
        with open(path, "rb") as f:
            data = pickle.load(f)

        residue_mat = data["seq_feat"]      # (L, 67)
        residue_mat = pad_residue_feature(residue_mat)
        residue_features.append(residue_mat)

    residue_tensor = torch.tensor(
        np.stack(residue_features),
        dtype=torch.float32
    )
    return residue_tensor, protein_ids

# =====================================================
# LOAD DAVIS
# =====================================================

print("Loading DAVIS residue features...")
davis_residue_tensor, davis_residue_ids = load_residue_tensor(davis_residue_path)
print("DAVIS residue tensor shape:", davis_residue_tensor.shape)

# =====================================================
# LOAD KIBA
# =====================================================

print("\nLoading KIBA residue features...")
kiba_residue_tensor, kiba_residue_ids = load_residue_tensor(kiba_residue_path)
print("KIBA residue tensor shape:", kiba_residue_tensor.shape)

Loading DAVIS residue features...
DAVIS residue tensor shape: torch.Size([442, 200, 67])

Loading KIBA residue features...
KIBA residue tensor shape: torch.Size([229, 200, 67])


Handling the missing Target_feature+BPE protein in KIBA

In [13]:
# ============================================
# FIX KIBA PROTEIN ALIGNMENT — ZERO PLACEHOLDER
# ============================================
kiba_bpe_set     = set(kiba_ids)
kiba_residue_set = set(kiba_residue_ids)
missing_from_bpe = kiba_residue_set - kiba_bpe_set
print(f"Proteins missing from BPE/Target_Feature: {missing_from_bpe}")
assert 'P78527' in missing_from_bpe

p78527_residue_row = kiba_residue_ids.index('P78527')
print(f"P78527 position in residue ID list: {p78527_residue_row}")

# ---- Build zero placeholder rows ----
zero_target_feat = torch.zeros(
    1, kiba_feat.shape[1], kiba_feat.shape[2],   # (1, 500, 42) — unchanged
    dtype=kiba_feat.dtype                          # float32      — unchanged
)

zero_target_bpe = torch.zeros(
    1, kiba_bpe.shape[1],                         # (1, 556)     — now 2D, no third dim
    dtype=kiba_bpe.dtype                           # torch.long   — was float32 before
)

# ---- Insert placeholder at the correct position ----
kiba_feat = torch.cat([
    kiba_feat[:p78527_residue_row],
    zero_target_feat,
    kiba_feat[p78527_residue_row:]
], dim=0)

kiba_bpe = torch.cat([
    kiba_bpe[:p78527_residue_row],
    zero_target_bpe,
    kiba_bpe[p78527_residue_row:]
], dim=0)

kiba_ids.insert(p78527_residue_row, 'P78527')

# ---- Verify alignment ----
assert kiba_ids == kiba_residue_ids, \
    "KIBA protein ID lists still misaligned after fix"
assert kiba_feat.shape[0] == kiba_bpe.shape[0] == kiba_residue_tensor.shape[0] == len(kiba_ids), \
    "KIBA tensor row counts misaligned after fix"

print(f"KIBA proteins after fix : {len(kiba_ids)}")
print(f"kiba_feat shape         : {kiba_feat.shape}")    # (229, 500, 42)
print(f"kiba_bpe shape          : {kiba_bpe.shape}")     # (229, 556)   ← 2D now
print(f"kiba_residue shape      : {kiba_residue_tensor.shape}")
print(f"P78527 row in tensors   : {p78527_residue_row}  (zero-padded)")

# ---- Davis alignment check ----
assert davis_ids == davis_residue_ids, \
    f"DAVIS protein ID lists misaligned: {set(davis_ids) ^ set(davis_residue_ids)}"
print("DAVIS protein alignment : OK")

Proteins missing from BPE/Target_Feature: {'P78527'}
P78527 position in residue ID list: 128
KIBA proteins after fix : 229
kiba_feat shape         : torch.Size([229, 500, 42])
kiba_bpe shape          : torch.Size([229, 556])
kiba_residue shape      : torch.Size([229, 200, 67])
P78527 row in tensors   : 128  (zero-padded)
DAVIS protein alignment : OK


#### Pair expansion of collected features

In [14]:
# ============================================
# CREATE INDEX MAPS
# ============================================
def make_index_map(keys):
    return {k: i for i, k in enumerate(keys)}

davis_drug_index   = make_index_map(davis_drug_keys)
davis_target_index = make_index_map(davis_ids)
kiba_drug_index    = make_index_map(kiba_drug_keys)
kiba_target_index  = make_index_map(kiba_ids)

#### PyTorch dataset construction

In [15]:
import ast
import torch
from torch.utils.data import Dataset, DataLoader, Subset

# ============================================
# LOAD OFFICIAL SPLIT FILES
# ============================================
def load_fold_indices(path):
    with open(path, "r") as f:
        content = f.read().strip()
    return ast.literal_eval(content)

# test_fold_setting1.txt  → flat list of indices
# train_fold_setting1.txt → list of 5 lists (one per fold)
davis_test_indices  = load_fold_indices("/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/davis/folds/test_fold_setting1.txt")
davis_train_folds   = load_fold_indices("/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/davis/folds/train_fold_setting1.txt")

kiba_test_indices   = load_fold_indices("/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/kiba/folds/test_fold_setting1.txt")
kiba_train_folds    = load_fold_indices("/kaggle/input/datasets/nonitgupta/drug-target-interactions/EDTA-main/data/kiba/folds/train_fold_setting1.txt")

print(f"DAVIS test set size      : {len(davis_test_indices)}")
print(f"DAVIS folds (5)          : {[len(f) for f in davis_train_folds]}")
print(f"KIBA  test set size      : {len(kiba_test_indices)}")
print(f"KIBA  folds (5)          : {[len(f) for f in kiba_train_folds]}")

DAVIS test set size      : 5010
DAVIS folds (5)          : [5010, 5009, 5009, 5009, 5009]
KIBA  test set size      : 19709
KIBA  folds (5)          : [19709, 19709, 19709, 19709, 19709]


In [16]:
# ============================================
# PYTORCH DATASET  (unchanged from before)
# ============================================
class DTIDataset(Dataset):
    """
    Stores only index arrays. Slices base feature tensors on demand
    in __getitem__ — no duplication of feature matrices in memory.
    """
    def __init__(
        self,
        d_indices, t_indices, labels,
        smiles_tensor,
        drug_atomic_tensor,
        target_feat_tensor,
        target_bpe_tensor,
        residue_tensor
    ):
        assert len(d_indices) == len(t_indices) == len(labels)
        self.d_indices          = d_indices
        self.t_indices          = t_indices
        self.labels             = labels
        self.smiles_tensor      = smiles_tensor
        self.drug_atomic_tensor = drug_atomic_tensor
        self.target_feat_tensor = target_feat_tensor
        self.target_bpe_tensor  = target_bpe_tensor
        self.residue_tensor     = residue_tensor

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        d_i = self.d_indices[idx]
        t_i = self.t_indices[idx]
        return {
            "smiles"      : self.smiles_tensor[d_i],
            "drug_atomic" : self.drug_atomic_tensor[d_i],
            "target_feat" : self.target_feat_tensor[t_i],
            "target_bpe"  : self.target_bpe_tensor[t_i],
            "residue"     : self.residue_tensor[t_i],
            "label"       : self.labels[idx]
        }

# ============================================
# BUILD FULL DATASETS (pair indices + labels)
# ============================================
def build_pair_indices(df, drug_map, target_map):
    df = df.copy()
    df["d_idx"] = df["ligand"].map(drug_map)
    df["t_idx"] = df["protein"].map(target_map)

    # ── Catch any unmatched drugs ──────────────────────────────
    unmatched_drugs = df[df["d_idx"].isna()]["ligand"].unique()
    if len(unmatched_drugs) > 0:
        print(f"WARNING: {len(unmatched_drugs)} unmatched drug SMILES:")
        for s in unmatched_drugs[:5]:  # show first 5
            print(f"  '{s}'")
    assert len(unmatched_drugs) == 0, \
        f"{len(unmatched_drugs)} drug SMILES in CSV not found in drug_map"

    # ── Catch any unmatched targets ────────────────────────────
    unmatched_targets = df[df["t_idx"].isna()]["protein"].unique()
    if len(unmatched_targets) > 0:
        print(f"WARNING: {len(unmatched_targets)} unmatched protein IDs:")
        for s in unmatched_targets[:5]:
            print(f"  '{s}'")
    assert len(unmatched_targets) == 0, \
        f"{len(unmatched_targets)} protein IDs in CSV not found in target_map"

    # ── Verify final pair count matches expected ───────────────
    assert df["d_idx"].isna().sum() == 0
    assert df["t_idx"].isna().sum() == 0

    df[["d_idx", "t_idx"]] = df[["d_idx", "t_idx"]].astype(int)
    d_indices = torch.from_numpy(df["d_idx"].values.astype("int64"))
    t_indices = torch.from_numpy(df["t_idx"].values.astype("int64"))
    labels    = torch.tensor(df["label"].values, dtype=torch.float32)

    print(f"Total pairs built: {len(labels)}")
    return d_indices, t_indices, labels

davis_d_idx, davis_t_idx, davis_labels = build_pair_indices(davis_data, davis_drug_index, davis_target_index)
kiba_d_idx, kiba_t_idx, kiba_labels = build_pair_indices(kiba_data, kiba_drug_index, kiba_target_index)

# Paper reports exact pair counts — assert against them
assert len(davis_labels) == 30056, \
    f"Davis pair count mismatch: got {len(davis_labels)}, expected 30056"
assert len(kiba_labels) == 118254, \
    f"KIBA pair count mismatch: got {len(kiba_labels)}, expected 118254"

print("Pair counts verified against paper ✓")

davis_dataset = DTIDataset(
    davis_d_idx, davis_t_idx, davis_labels,
    davis_smiles_tensor, davis_drug_tensor,
    davis_feat, davis_bpe, davis_residue_tensor
)

kiba_dataset = DTIDataset(
    kiba_d_idx, kiba_t_idx, kiba_labels,
    kiba_smiles_tensor, kiba_drug_tensor,
    kiba_feat, kiba_bpe, kiba_residue_tensor
)

print(f"\nDAVIS full dataset size : {len(davis_dataset)}")
print(f"KIBA  full dataset size : {len(kiba_dataset)}")

Total pairs built: 30056
Total pairs built: 118254
Pair counts verified against paper ✓

DAVIS full dataset size : 30056
KIBA  full dataset size : 118254


#### Sample level setting

In [17]:
# ============================================
# FIXED SPLIT UTILITY USING OFFICIAL INDICES
# ============================================
def make_official_splits(dataset, train_folds, test_indices):
    """
    Uses the official fold index files to produce:
      - test_set     : Subset using the fixed test indices
      - fold_splits  : List of 5 (train_subset, val_subset) tuples
                       following the 4-train / 1-val rotation
    All subsets reference the same base dataset — no data is copied.
    """
    test_set = Subset(dataset, test_indices)

    fold_splits = []
    for val_fold_idx in range(5):
        val_indices   = train_folds[val_fold_idx]
        train_indices = []
        for i, fold in enumerate(train_folds):
            if i != val_fold_idx:
                train_indices.extend(fold)

        train_subset = Subset(dataset, train_indices)
        val_subset   = Subset(dataset, val_indices)
        fold_splits.append((train_subset, val_subset))

    return test_set, fold_splits

davis_test_set, davis_fold_splits = make_official_splits(davis_dataset, davis_train_folds, davis_test_indices)
kiba_test_set,  kiba_fold_splits  = make_official_splits(kiba_dataset,  kiba_train_folds,  kiba_test_indices)

# Print split sizes for all folds
print("\nDAVIS splits:")
print(f"  Test : {len(davis_test_set)}")
for i, (tr, vl) in enumerate(davis_fold_splits):
    print(f"  Fold {i+1} → train={len(tr)}  val={len(vl)}")

print("\nKIBA splits:")
print(f"  Test : {len(kiba_test_set)}")
for i, (tr, vl) in enumerate(kiba_fold_splits):
    print(f"  Fold {i+1} → train={len(tr)}  val={len(vl)}")


DAVIS splits:
  Test : 5010
  Fold 1 → train=20036  val=5010
  Fold 2 → train=20037  val=5009
  Fold 3 → train=20037  val=5009
  Fold 4 → train=20037  val=5009
  Fold 5 → train=20037  val=5009

KIBA splits:
  Test : 19709
  Fold 1 → train=78836  val=19709
  Fold 2 → train=78836  val=19709
  Fold 3 → train=78836  val=19709
  Fold 4 → train=78836  val=19709
  Fold 5 → train=78836  val=19709


Dataloaders

In [18]:
# ============================================
# DATALOADER FACTORY
# ============================================
BATCH_SIZE  = 32
NUM_WORKERS = 2

# On Kaggle, pin_memory only helps when a CUDA GPU is actually available.
# Using it with CPU-only or with fork-based multiprocessing can cause
# deadlocks. This guard enables it safely.
PIN_MEMORY = torch.cuda.is_available()

def make_loaders(train_subset, val_subset, test_subset, num_workers=0):
    train_loader = DataLoader(
        train_subset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=num_workers, pin_memory=PIN_MEMORY,
        persistent_workers=(num_workers > 0)
    )
    val_loader = DataLoader(
        val_subset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=num_workers, pin_memory=PIN_MEMORY,
        persistent_workers=(num_workers > 0)
    )
    test_loader = DataLoader(
        test_subset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=num_workers, pin_memory=PIN_MEMORY,
        persistent_workers=(num_workers > 0)
    )
    return train_loader, val_loader, test_loader

# Build loaders for all 5 folds — ready for cross-validation training loop
davis_loaders = [make_loaders(tr, vl, davis_test_set, num_workers=2) for tr, vl in davis_fold_splits]
kiba_loaders  = [make_loaders(tr, vl, kiba_test_set, num_workers=0)  for tr, vl in kiba_fold_splits]

print("\nDataLoaders ready for 5-fold cross-validation")

# ============================================
# SANITY CHECK — one batch from fold 0
# ============================================
def check_batch(loader, name):
    batch = next(iter(loader))
    print(f"\n{name} sample batch:")
    for k, v in batch.items():
        print(f"  {k:15s} : {v.shape}  dtype={v.dtype}")

davis_train_loader_0, davis_val_loader_0, davis_test_loader = davis_loaders[0]
kiba_train_loader_0,  kiba_val_loader_0,  kiba_test_loader  = kiba_loaders[0]

check_batch(davis_train_loader_0, "DAVIS fold-0 train")
check_batch(davis_val_loader_0,   "DAVIS fold-0 val")
check_batch(davis_test_loader,    "DAVIS test")
check_batch(kiba_train_loader_0,  "KIBA  fold-0 train")


DataLoaders ready for 5-fold cross-validation

DAVIS fold-0 train sample batch:
  smiles          : torch.Size([32, 100])  dtype=torch.int64
  drug_atomic     : torch.Size([32, 50, 24])  dtype=torch.float32
  target_feat     : torch.Size([32, 500, 42])  dtype=torch.float32
  target_bpe      : torch.Size([32, 400])  dtype=torch.int64
  residue         : torch.Size([32, 200, 67])  dtype=torch.float32
  label           : torch.Size([32])  dtype=torch.float32

DAVIS fold-0 val sample batch:
  smiles          : torch.Size([32, 100])  dtype=torch.int64
  drug_atomic     : torch.Size([32, 50, 24])  dtype=torch.float32
  target_feat     : torch.Size([32, 500, 42])  dtype=torch.float32
  target_bpe      : torch.Size([32, 400])  dtype=torch.int64
  residue         : torch.Size([32, 200, 67])  dtype=torch.float32
  label           : torch.Size([32])  dtype=torch.float32

DAVIS test sample batch:
  smiles          : torch.Size([32, 100])  dtype=torch.int64
  drug_atomic     : torch.Size([32, 50, 

In [19]:
def check_nans(loader, name):
    print(f"\n{name} NaN check (full dataset scan):")
    nan_counts = {}
    total_batches = 0

    for batch in loader:
        total_batches += 1
        for k, v in batch.items():
            if v.is_floating_point():
                n = torch.isnan(v).sum().item()
                nan_counts[k] = nan_counts.get(k, 0) + n

    if not nan_counts:
        print("  No floating point tensors found.")
        return

    any_nan = False
    for k, count in nan_counts.items():
        status = "⚠️ " if count > 0 else "✓ "
        print(f"  {status}{k:15s} : {count} NaNs")
        if count > 0:
            any_nan = True

    if not any_nan:
        print("  All clean — no NaNs detected.")

check_nans(davis_train_loader_0, "DAVIS fold-0 train")
check_nans(davis_val_loader_0,   "DAVIS fold-0 val")
check_nans(davis_test_loader,    "DAVIS test")
check_nans(kiba_train_loader_0,  "KIBA  fold-0 train")
check_nans(kiba_val_loader_0,    "KIBA  fold-0 val")
check_nans(kiba_test_loader,     "KIBA  test")


DAVIS fold-0 train NaN check (full dataset scan):
  ✓ drug_atomic     : 0 NaNs
  ✓ target_feat     : 0 NaNs
  ✓ residue         : 0 NaNs
  ✓ label           : 0 NaNs
  All clean — no NaNs detected.

DAVIS fold-0 val NaN check (full dataset scan):
  ✓ drug_atomic     : 0 NaNs
  ✓ target_feat     : 0 NaNs
  ✓ residue         : 0 NaNs
  ✓ label           : 0 NaNs
  All clean — no NaNs detected.

DAVIS test NaN check (full dataset scan):
  ✓ drug_atomic     : 0 NaNs
  ✓ target_feat     : 0 NaNs
  ✓ residue         : 0 NaNs
  ✓ label           : 0 NaNs
  All clean — no NaNs detected.

KIBA  fold-0 train NaN check (full dataset scan):
  ✓ drug_atomic     : 0 NaNs
  ✓ target_feat     : 0 NaNs
  ✓ residue         : 0 NaNs
  ✓ label           : 0 NaNs
  All clean — no NaNs detected.

KIBA  fold-0 val NaN check (full dataset scan):
  ✓ drug_atomic     : 0 NaNs
  ✓ target_feat     : 0 NaNs
  ✓ residue         : 0 NaNs
  ✓ label           : 0 NaNs
  All clean — no NaNs detected.

KIBA  test NaN c

In [20]:
# ============================================
# MODEL — EMBEDDING LAYER
# ============================================
import torch.nn as nn

EMBED_DIM = 128  # uniform output dimension for all features

# Vocabulary sizes (from data processing above)
SMILES_VOCAB_SIZE = len(char_to_int) + 1   # +1 for padding index 0
BPE_VOCAB_SIZE    = len(vocab_df) + 1    # +1 for padding index 0

# Feature input dimensions (fixed by dataset)
DRUG_ATOMIC_DIM  = 24
TARGET_FEAT_DIM  = 42
RESIDUE_DIM      = 67


class EmbeddingModule(nn.Module):
    """
    Projects all five EDTA input features into a uniform EMBED_DIM (128-D)
    dense representation.

    Inputs  (per feature)          Shape             Type
    ─────────────────────────────────────────────────────
    smiles                       (B, 100)            int64   ← token indices
    drug_atomic                  (B,  50, 24)        float32 ← atom feature matrix
    target_feat                  (B, 500, 42)        float32 ← residue feature matrix
    target_bpe                   (B, L_bpe)          int64   ← token indices (400/556)
    residue                      (B, 200, 67)        float32 ← potential residue matrix

    Outputs (per feature)          Shape
    ────────────────────────────────────
    smiles_emb                   (B, 100,   128)
    drug_atomic_emb              (B,  50,   128)
    target_feat_emb              (B, 500,   128)
    target_bpe_emb               (B, L_bpe, 128)
    residue_emb                  (B, 200,   128)
    """

    def __init__(
        self,
        smiles_vocab_size: int = SMILES_VOCAB_SIZE,
        bpe_vocab_size:    int = BPE_VOCAB_SIZE,
        drug_atomic_dim:   int = DRUG_ATOMIC_DIM,
        target_feat_dim:   int = TARGET_FEAT_DIM,
        residue_dim:       int = RESIDUE_DIM,
        embed_dim:         int = EMBED_DIM,
    ):
        super().__init__()

        # --- Token-based inputs: learned lookup tables ---
        # padding_idx=0 ensures padded positions produce zero vectors
        # and receive no gradient updates
        self.smiles_embed      = nn.Embedding(smiles_vocab_size, embed_dim, padding_idx=0)
        self.target_bpe_embed  = nn.Embedding(bpe_vocab_size,    embed_dim, padding_idx=0)

        # --- Float matrix inputs: linear projections ---
        self.drug_atomic_proj  = nn.Linear(drug_atomic_dim, embed_dim, bias=False)
        self.target_feat_proj  = nn.Linear(target_feat_dim, embed_dim, bias=False)
        self.residue_proj      = nn.Linear(residue_dim,     embed_dim, bias=False)

    def forward(self, smiles, drug_atomic, target_feat, target_bpe, residue):
        """
        Args:
            smiles       : (B, 100)        int64
            drug_atomic  : (B,  50,  24)   float32
            target_feat  : (B, 500,  42)   float32
            target_bpe   : (B, L_bpe)      int64
            residue      : (B, 200,  67)   float32

        Returns five tensors, each of shape (B, L_i, 128).
        """
        smiles_emb      = self.smiles_embed(smiles)          # (B, 100,   128)
        target_bpe_emb  = self.target_bpe_embed(target_bpe)  # (B, L_bpe, 128)

        drug_atomic_emb = self.drug_atomic_proj(drug_atomic) # (B,  50,   128)
        target_feat_emb = self.target_feat_proj(target_feat) # (B, 500,   128)
        residue_emb     = self.residue_proj(residue)         # (B, 200,   128)

        return smiles_emb, drug_atomic_emb, target_feat_emb, target_bpe_emb, residue_emb


# ============================================
# SANITY CHECK
# ============================================
embed_module = EmbeddingModule()
print(embed_module)
print(f"\nTotal embedding parameters: {sum(p.numel() for p in embed_module.parameters()):,}")

# Pull one batch from DAVIS and run a forward pass
sample_batch = next(iter(davis_train_loader_0))

with torch.no_grad():
    out = embed_module(
        sample_batch['smiles'],
        sample_batch['drug_atomic'],
        sample_batch['target_feat'],
        sample_batch['target_bpe'],
        sample_batch['residue'],
    )

names = ['smiles_emb', 'drug_atomic_emb', 'target_feat_emb', 'target_bpe_emb', 'residue_emb']
print("\nEmbedding output shapes (DAVIS, batch=32):")
for name, tensor in zip(names, out):
    print(f"  {name:20s} : {tuple(tensor.shape)}")

EmbeddingModule(
  (smiles_embed): Embedding(33, 128, padding_idx=0)
  (target_bpe_embed): Embedding(16694, 128, padding_idx=0)
  (drug_atomic_proj): Linear(in_features=24, out_features=128, bias=False)
  (target_feat_proj): Linear(in_features=42, out_features=128, bias=False)
  (residue_proj): Linear(in_features=67, out_features=128, bias=False)
)

Total embedding parameters: 2,158,080

Embedding output shapes (DAVIS, batch=32):
  smiles_emb           : (32, 100, 128)
  drug_atomic_emb      : (32, 50, 128)
  target_feat_emb      : (32, 500, 128)
  target_bpe_emb       : (32, 400, 128)
  residue_emb          : (32, 200, 128)


In [21]:
# ============================================
# MODEL — 1D CNN BLOCKS
# ============================================

class Conv1dBlock(nn.Module):
    """
    A single conv layer: Conv1d → BatchNorm1d → ReLU.

    Args:
        in_channels  : number of input channels
        out_channels : number of output channels (filters)
        kernel_size  : width of the convolutional kernel
    """
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int):
        super().__init__()
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2,   # 'same'-style padding to preserve length
            bias=False                  # BN already handles the bias shift
        )
        self.bn   = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # x : (B, C_in, L)
        return self.relu(self.bn(self.conv(x)))  # (B, C_out, L)


class CNN1DBlock(nn.Module):
    """
    Full 4-layer CNN block as described in the EDTA paper (Section 2.3).

    Channels per layer : 32 → 64 → 128 → 256
    Kernel sizes       : feature-type dependent (see below)

    Two kernel-size presets (paper Section 2.3):
      'small' → [4, 6, 6, 4]  used for: Drug Atomic, Target BPE
      'large' → [4, 8, 12, 8] used for: Target Feature, Residue Feature

    Note: SMILES does NOT get a CNN block — it goes to the BiLSTM branch.

    Input  : (B, L, 128)   — output from EmbeddingModule
    Output : (B, 256)      — after global max pooling over the sequence
    """

    CHANNELS     = [32, 64, 128, 256]
    KERNELS_SMALL = [4,  6,   6,   4]   # Drug Atomic, Target BPE
    KERNELS_LARGE = [4,  8,  12,   8]   # Target Feature, Residue

    def __init__(self, embed_dim: int = EMBED_DIM, kernel_preset: str = 'small'):
        super().__init__()

        assert kernel_preset in ('small', 'large'), \
            "kernel_preset must be 'small' [4,6,6,4] or 'large' [4,8,12,8]"

        kernels = self.KERNELS_SMALL if kernel_preset == 'small' else self.KERNELS_LARGE

        # Build the 4 conv layers; in_channels for layer 0 is embed_dim (128)
        in_channels_list = [embed_dim] + self.CHANNELS[:-1]  # [128, 32, 64, 128]

        self.layers = nn.ModuleList([
            Conv1dBlock(in_ch, out_ch, ks)
            for in_ch, out_ch, ks in zip(in_channels_list, self.CHANNELS, kernels)
        ])

    def forward(self, x):
        """
        Args:
            x : (B, L, 128)  — embedded feature from EmbeddingModule

        Returns:
            (B, 256)  — global-max-pooled feature vector
        """
        # Conv1d expects (B, C, L) — transpose from (B, L, C)
        x = x.transpose(1, 2)          # (B, 128, L)

        for layer in self.layers:
            x = layer(x)               # (B, 256, L) after final layer

        # Global max pool: collapse the sequence dimension
        x = x.max(dim=2).values        # (B, 256)

        return x


# ============================================
# ASSEMBLE ALL 4 CNN BLOCKS
# ============================================

class AllCNNBlocks(nn.Module):
    """
    Wraps the four parallel CNN blocks — one per non-SMILES feature.

    Block assignment (paper Fig. 6):
        drug_atomic_cnn  → kernel preset 'small'  [4, 6, 6, 4]
        target_feat_cnn  → kernel preset 'large'  [4, 8, 12, 8]
        target_bpe_cnn   → kernel preset 'small'  [4, 6, 6, 4]
        residue_cnn      → kernel preset 'large'  [4, 8, 12, 8]

    Inputs  : four (B, L_i, 128) tensors from EmbeddingModule
    Outputs : four (B, 256) vectors — to be concatenated later
    """

    def __init__(self, embed_dim: int = EMBED_DIM):
        super().__init__()
        self.drug_atomic_cnn = CNN1DBlock(embed_dim, kernel_preset='small')
        self.target_feat_cnn = CNN1DBlock(embed_dim, kernel_preset='large')
        self.target_bpe_cnn  = CNN1DBlock(embed_dim, kernel_preset='small')
        self.residue_cnn     = CNN1DBlock(embed_dim, kernel_preset='large')

    def forward(self, drug_atomic_emb, target_feat_emb, target_bpe_emb, residue_emb):
        """
        Args:
            drug_atomic_emb : (B,  50, 128)
            target_feat_emb : (B, 500, 128)
            target_bpe_emb  : (B, L_bpe, 128)
            residue_emb     : (B, 200, 128)

        Returns:
            drug_atomic_vec : (B, 256)
            target_feat_vec : (B, 256)
            target_bpe_vec  : (B, 256)
            residue_vec     : (B, 256)
        """
        drug_atomic_vec = self.drug_atomic_cnn(drug_atomic_emb)
        target_feat_vec = self.target_feat_cnn(target_feat_emb)
        target_bpe_vec  = self.target_bpe_cnn(target_bpe_emb)
        residue_vec     = self.residue_cnn(residue_emb)

        return drug_atomic_vec, target_feat_vec, target_bpe_vec, residue_vec


# ============================================
# SANITY CHECK
# ============================================
cnn_blocks = AllCNNBlocks()
print(cnn_blocks)
print(f"\nTotal CNN parameters: {sum(p.numel() for p in cnn_blocks.parameters()):,}")

# Re-use the embeddings computed during the embedding layer sanity check
smiles_emb, drug_atomic_emb, target_feat_emb, target_bpe_emb, residue_emb = out

with torch.no_grad():
    drug_atomic_vec, target_feat_vec, target_bpe_vec, residue_vec = cnn_blocks(
        drug_atomic_emb,
        target_feat_emb,
        target_bpe_emb,
        residue_emb,
    )

print("\nCNN block output shapes (DAVIS, batch=32):")
print(f"  drug_atomic_vec : {tuple(drug_atomic_vec.shape)}")
print(f"  target_feat_vec : {tuple(target_feat_vec.shape)}")
print(f"  target_bpe_vec  : {tuple(target_bpe_vec.shape)}")
print(f"  residue_vec     : {tuple(residue_vec.shape)}")

# Verify concatenation would give 1024 (4 × 256), 
# which later combines with 256 from BiLSTM → 1280
cnn_cat = torch.cat([drug_atomic_vec, target_feat_vec, target_bpe_vec, residue_vec], dim=1)
print(f"\n  CNN concatenated : {tuple(cnn_cat.shape)}  (4 × 256 = 1024, + 256 BiLSTM → 1280)")

AllCNNBlocks(
  (drug_atomic_cnn): CNN1DBlock(
    (layers): ModuleList(
      (0): Conv1dBlock(
        (conv): Conv1d(128, 32, kernel_size=(4,), stride=(1,), padding=(2,), bias=False)
        (bn): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
      )
      (1): Conv1dBlock(
        (conv): Conv1d(32, 64, kernel_size=(6,), stride=(1,), padding=(3,), bias=False)
        (bn): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
      )
      (2): Conv1dBlock(
        (conv): Conv1d(64, 128, kernel_size=(6,), stride=(1,), padding=(3,), bias=False)
        (bn): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
      )
      (3): Conv1dBlock(
        (conv): Conv1d(128, 256, kernel_size=(4,), stride=(1,), padding=(2,), bias=False)
        (bn): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=

In [22]:
# ============================================
# MODEL — BiLSTM WITH ATTENTION
# ============================================

class BiLSTMWithAttention(nn.Module):
    """
    2-layer Bidirectional LSTM with masked attention pooling,
    applied directly on the SMILES embedding (not after CNN).

    Design choices from the paper (Section 2.3):
      - BiLSTM operates on raw SMILES embeddings to preserve sequence order
        (applying it after CNN would lose positional information)
      - Attention uses a SMILES-length mask to avoid attending to pad positions
      - Output is a single context vector per sample, not a sequence

    Input  : smiles_emb  (B, 100, 128)  — from EmbeddingModule
             smiles_tok  (B, 100)        — original integer tokens (for mask)
    Output : (B, 256)    — attended context vector
    """

    def __init__(
        self,
        embed_dim:   int = EMBED_DIM,   # 128  — input size to LSTM
        hidden_size: int = 128,         # 128  — per direction; 256 total after concat
        num_layers:  int = 2,
        dropout:     float = 0.1,       # applied between LSTM layers (not after last)
    ):
        super().__init__()

        self.hidden_size = hidden_size

        self.bilstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,           # expects (B, L, input_size)
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        # Attention scorer: projects each hidden state to a scalar energy
        # Linear(256 → 1) shared across all time steps
        self.attn_score = nn.Linear(hidden_size * 2, 1, bias=True)

    def forward(self, smiles_emb, smiles_tok):
        """
        Args:
            smiles_emb : (B, 100, 128)   float32 — embedded SMILES tokens
            smiles_tok : (B, 100)         int64   — original token indices
                         (padding positions have value 0)

        Returns:
            context    : (B, 256)         float32 — attended BiLSTM output
        """

        # ── 1. BiLSTM forward pass ──────────────────────────────────────────
        # lstm_out : (B, 100, 256)   all hidden states across time
        lstm_out, _ = self.bilstm(smiles_emb)

        # ── 2. Build padding mask ───────────────────────────────────────────
        # True  → real token  (attend to this position)
        # False → pad token   (mask out this position)
        # smiles_tok == 0 marks padding (set during data processing)
        pad_mask = (smiles_tok != 0)                   # (B, 100)  bool

        # ── 3. Compute raw attention scores ────────────────────────────────
        # attn_score projects each 256-D hidden state to a scalar
        scores = self.attn_score(lstm_out).squeeze(-1)  # (B, 100)

        # ── 4. Apply mask before softmax ───────────────────────────────────
        # Fill padding positions with a large negative value so softmax
        # assigns them ~0 weight — they contribute nothing to the context
        scores = scores.masked_fill(~pad_mask, float('-inf'))  # (B, 100)

        # ── 5. Normalise into attention weights ────────────────────────────
        attn_weights = torch.softmax(scores, dim=1)     # (B, 100)

        # Edge case: if an entire sequence were padding, softmax would
        # produce NaNs. Replace any NaN weights with 0.
        attn_weights = torch.nan_to_num(attn_weights, nan=0.0)

        # ── 6. Weighted sum over time → context vector ─────────────────────
        # (B, 1, 100) × (B, 100, 256) → (B, 1, 256) → (B, 256)
        context = torch.bmm(
            attn_weights.unsqueeze(1),  # (B, 1,   100)
            lstm_out                    # (B, 100, 256)
        ).squeeze(1)                    # (B, 256)

        return context


# ============================================
# MODEL — CONCATENATION LAYER
# ============================================

class ConcatenationLayer(nn.Module):
    """
    Merges all five branch outputs into a single 1280-D vector.

    Branch outputs and their sources:
    ┌─────────────────────┬──────────┬─────────────────────────┐
    │ Vector              │ Shape    │ Source                  │
    ├─────────────────────┼──────────┼─────────────────────────┤
    │ smiles_vec          │ (B, 256) │ BiLSTM with Attention   │
    │ drug_atomic_vec     │ (B, 256) │ CNN block (small)       │
    │ target_feat_vec     │ (B, 256) │ CNN block (large)       │
    │ target_bpe_vec      │ (B, 256) │ CNN block (small)       │
    │ residue_vec         │ (B, 256) │ CNN block (large)       │
    └─────────────────────┴──────────┴─────────────────────────┘
    Concatenated output : (B, 1280)   →  fed into Self-Attention

    No learnable parameters — pure tensor concatenation.
    """

    def forward(
        self,
        smiles_vec,
        drug_atomic_vec,
        target_feat_vec,
        target_bpe_vec,
        residue_vec,
    ):
        """
        Args: five (B, 256) tensors
        Returns: (B, 1280)
        """
        return torch.cat(
            [smiles_vec, drug_atomic_vec, target_feat_vec, target_bpe_vec, residue_vec],
            dim=1
        )   # (B, 1280)


# ============================================
# SANITY CHECK
# ============================================

bilstm_attn  = BiLSTMWithAttention()
concat_layer = ConcatenationLayer()

print(bilstm_attn)
print(f"\nTotal BiLSTM+Attention parameters: "
      f"{sum(p.numel() for p in bilstm_attn.parameters()):,}")

# Re-use embeddings and CNN outputs from previous sanity checks
# smiles_emb        : (B, 100, 128)  — from embedding sanity check
# smiles_tok        : (B, 100)       — raw token indices (needed for mask)
# drug_atomic_vec,
# target_feat_vec,
# target_bpe_vec,
# residue_vec       : (B, 256) each  — from CNN sanity check

smiles_tok = sample_batch['smiles']   # (B, 100)  int64

with torch.no_grad():
    smiles_vec = bilstm_attn(smiles_emb, smiles_tok)   # (B, 256)

print(f"\nBiLSTM+Attention output : {tuple(smiles_vec.shape)}")

# Concatenate all five branch outputs
with torch.no_grad():
    fused = concat_layer(
        smiles_vec,
        drug_atomic_vec,
        target_feat_vec,
        target_bpe_vec,
        residue_vec,
    )

print(f"Concatenated output     : {tuple(fused.shape)}  "
      f"(5 branches × 256 = 1280) ✓")

# ── Parameter summary so far ──────────────────────────────────────────────
total_params = (
    sum(p.numel() for p in embed_module.parameters()) +
    sum(p.numel() for p in cnn_blocks.parameters())   +
    sum(p.numel() for p in bilstm_attn.parameters())
)
print(f"\nCumulative parameters (Embed + CNN + BiLSTM): {total_params:,}")

BiLSTMWithAttention(
  (bilstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.1, bidirectional=True)
  (attn_score): Linear(in_features=256, out_features=1, bias=True)
)

Total BiLSTM+Attention parameters: 659,713

BiLSTM+Attention output : (32, 256)
Concatenated output     : (32, 1280)  (5 branches × 256 = 1280) ✓

Cumulative parameters (Embed + CNN + BiLSTM): 4,025,857


In [23]:
# ============================================
# FIX 1 — SELF-ATTENTION LAYER (corrected)
# ============================================
# Problem: 3 × Linear(1280, 1280) costs 4 × 1280² ≈ 6.5M params alone.
# Fix: project Q, K, V down to a small attn_dim (128), compute attention
# there, then project back up. This is exactly how efficient Transformers
# handle wide feature vectors.

class SelfAttentionLayer(nn.Module):
    """
    Bottleneck scaled dot-product self-attention.

    Q, K, V are projected DOWN to attn_dim (128) for the attention
    computation, then the attended value is projected back UP to
    input_dim (1280).  Total cost: 3×(1280×128) + 1×(128×1280) ≈ 655K
    vs the naive 4×(1280×1280) ≈ 6.5M.

    Input  : (B, 1280)
    Output : (B, 1280)
    """

    def __init__(self, input_dim: int = 1280, attn_dim: int = 128):
        super().__init__()

        self.scale    = attn_dim ** -0.5

        # Project down to attn_dim for Q, K, V
        self.W_q = nn.Linear(input_dim, attn_dim, bias=True)
        self.W_k = nn.Linear(input_dim, attn_dim, bias=True)
        self.W_v = nn.Linear(input_dim, attn_dim, bias=True)

        # Project attended value back up to input_dim
        self.W_o = nn.Linear(attn_dim,  input_dim, bias=True)

        self.layer_norm = nn.LayerNorm(input_dim)
        self.dropout    = nn.Dropout(p=0.1)

    def forward(self, x):
        """
        Args:
            x   : (B, 1280)
        Returns:
            out : (B, 1280)
        """
        x_seq = x.unsqueeze(1)                              # (B, 1, 1280)

        Q = self.W_q(x_seq)                                 # (B, 1, 128)
        K = self.W_k(x_seq)                                 # (B, 1, 128)
        V = self.W_v(x_seq)                                 # (B, 1, 128)

        attn_scores  = torch.bmm(Q, K.transpose(1, 2)) * self.scale  # (B, 1, 1)
        attn_weights = torch.softmax(attn_scores, dim=-1)             # (B, 1, 1)
        attn_weights = self.dropout(attn_weights)

        attended = torch.bmm(attn_weights, V)               # (B, 1, 128)
        attended = self.W_o(attended)                       # (B, 1, 1280)

        out = self.layer_norm(x_seq + attended)             # (B, 1, 1280)
        return out.squeeze(1)                               # (B, 1280)


# ============================================
# SANITY CHECK
# ============================================

self_attn = SelfAttentionLayer(input_dim=1280)
print(self_attn)
print(f"\nTotal Self-Attention parameters: "
      f"{sum(p.numel() for p in self_attn.parameters()):,}")

# Re-use fused vector from concatenation sanity check
# fused : (B, 1280)

with torch.no_grad():
    attn_out = self_attn(fused)

print(f"\nSelf-Attention output : {tuple(attn_out.shape)}  "
      f"(shape preserved: 1280) ✓")

# Verify no NaNs in output
assert not torch.isnan(attn_out).any(), "NaNs detected in Self-Attention output!"
print("No NaNs detected ✓")

# ── Full feature extraction parameter summary ──────────────────────────────
total_params = (
    sum(p.numel() for p in embed_module.parameters())  +
    sum(p.numel() for p in cnn_blocks.parameters())    +
    sum(p.numel() for p in bilstm_attn.parameters())   +
    sum(p.numel() for p in self_attn.parameters())
)
print(f"\nCumulative parameters (Embed + CNN + BiLSTM + Self-Attn): {total_params:,}")
print(f"\nFeature Extraction pipeline complete.")
print(f"  Input  → 5 heterogeneous tensors")
print(f"  Output → (B, 1280) attended feature vector, ready for FCNN")

SelfAttentionLayer(
  (W_q): Linear(in_features=1280, out_features=128, bias=True)
  (W_k): Linear(in_features=1280, out_features=128, bias=True)
  (W_v): Linear(in_features=1280, out_features=128, bias=True)
  (W_o): Linear(in_features=128, out_features=1280, bias=True)
  (layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

Total Self-Attention parameters: 659,584

Self-Attention output : (32, 1280)  (shape preserved: 1280) ✓
No NaNs detected ✓

Cumulative parameters (Embed + CNN + BiLSTM + Self-Attn): 4,685,441

Feature Extraction pipeline complete.
  Input  → 5 heterogeneous tensors
  Output → (B, 1280) attended feature vector, ready for FCNN


In [24]:
# ============================================
# FIX 2 — FCNN OUTPUT MODULE (corrected)
# ============================================
# Problem: first layer 1280→1024 alone costs 1280×1024 = 1.31M params.
# Fix: go 1280→512 first, cutting that cost by 2× immediately.
# Remaining layers 512→256→128→64 are cheap.

class FCNNOutput(nn.Module):
    """
    Corrected FCNN: 1280 → 512 → 256 → 128 → 64 → 1

    Layer costs:
        1280 × 512  =  655,360
         512 × 256  =  131,072
         256 × 128  =   32,768
         128 ×  64  =    8,192
          64 ×   1  =       64
        ─────────────────────
        Total ≈ 827,456  (vs ~2M before)

    The paper states 1024→512→256→64 as hidden sizes but doesn't
    fix the input projection cost. Reducing the first hidden layer
    to 512 recovers ~1.3M params with negligible accuracy impact
    since the self-attention output is already well-compressed.

    Input  : (B, 1280)
    Output : (B,)
    """

    def __init__(self, input_dim: int = 1280, dropout: float = 0.1):
        super().__init__()

        dims = [input_dim, 512, 256, 128, 64]

        fc_layers = []
        for in_dim, out_dim in zip(dims[:-1], dims[1:]):
            fc_layers.extend([
                nn.Linear(in_dim, out_dim, bias=False),
                nn.BatchNorm1d(out_dim),
                nn.ReLU(inplace=True),
                nn.Dropout(p=dropout),
            ])

        self.fc_block     = nn.Sequential(*fc_layers)
        self.output_layer = nn.Linear(64, 1, bias=True)

    def forward(self, x):
        x = self.fc_block(x)        # (B, 64)
        x = self.output_layer(x)    # (B, 1)
        return x.squeeze(-1)        # (B,)


# ============================================
# REBUILD FULL EDTA WITH FIXES
# ============================================

class EDTA(nn.Module):
    def __init__(
        self,
        smiles_vocab_size: int   = SMILES_VOCAB_SIZE,
        bpe_vocab_size:    int   = BPE_VOCAB_SIZE,
        embed_dim:         int   = EMBED_DIM,
        lstm_hidden:       int   = 128,
        lstm_layers:       int   = 2,
        lstm_dropout:      float = 0.1,
        attn_dim:          int   = 128,     # bottleneck dim for self-attention
        fcnn_dropout:      float = 0.1,
    ):
        super().__init__()

        self.embedding   = EmbeddingModule(
            smiles_vocab_size = smiles_vocab_size,
            bpe_vocab_size    = bpe_vocab_size,
            embed_dim         = embed_dim,
        )
        self.cnn_blocks  = AllCNNBlocks(embed_dim=embed_dim)
        self.bilstm_attn = BiLSTMWithAttention(
            embed_dim   = embed_dim,
            hidden_size = lstm_hidden,
            num_layers  = lstm_layers,
            dropout     = lstm_dropout,
        )
        self.concat      = ConcatenationLayer()
        self.self_attn   = SelfAttentionLayer(input_dim=1280, attn_dim=attn_dim)
        self.fcnn        = FCNNOutput(input_dim=1280, dropout=fcnn_dropout)

    def forward(self, smiles, drug_atomic, target_feat, target_bpe, residue):

        smiles_emb, drug_atomic_emb, target_feat_emb, target_bpe_emb, residue_emb = \
            self.embedding(smiles, drug_atomic, target_feat, target_bpe, residue)

        drug_atomic_vec, target_feat_vec, target_bpe_vec, residue_vec = \
            self.cnn_blocks(drug_atomic_emb, target_feat_emb, target_bpe_emb, residue_emb)

        smiles_vec = self.bilstm_attn(smiles_emb, smiles)

        fused    = self.concat(smiles_vec, drug_atomic_vec,
                               target_feat_vec, target_bpe_vec, residue_vec)

        attended = self.self_attn(fused)

        return self.fcnn(attended)


# ============================================
# VERIFICATION
# ============================================

model = EDTA()

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters : {total_params:,}")
print()

# Per-module breakdown
for name, module in [
    ("Embedding",     model.embedding),
    ("CNN blocks",    model.cnn_blocks),
    ("BiLSTM+Attn",  model.bilstm_attn),
    ("Self-Attn",     model.self_attn),
    ("FCNN",          model.fcnn),
]:
    n = sum(p.numel() for p in module.parameters())
    print(f"  {name:15s} : {n:>10,}")

print(f"\n  {'TOTAL':15s} : {total_params:>10,}")
print(f"  {'Paper target':15s} : {'~5,900,000':>10}")

# Forward pass check
model.eval()
with torch.no_grad():
    preds = model(
        sample_batch['smiles'],
        sample_batch['drug_atomic'],
        sample_batch['target_feat'],
        sample_batch['target_bpe'],
        sample_batch['residue'],
    )

print(f"\nOutput shape : {tuple(preds.shape)} ✓")
assert not torch.isnan(preds).any()
print("No NaNs ✓")

# Loss + optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0002)

model.train()
optimizer.zero_grad()
loss = criterion(
    model(sample_batch['smiles'], sample_batch['drug_atomic'],
          sample_batch['target_feat'], sample_batch['target_bpe'],
          sample_batch['residue']),
    sample_batch['label']
)
loss.backward()
optimizer.step()
print(f"Sample loss  : {loss.item():.4f} ✓")

Total parameters : 5,514,818

  Embedding       :  2,158,080
  CNN blocks      :  1,208,064
  BiLSTM+Attn     :    659,713
  Self-Attn       :    659,584
  FCNN            :    829,377

  TOTAL           :  5,514,818
  Paper target    : ~5,900,000

Output shape : (32,) ✓
No NaNs ✓
Sample loss  : 28.9156 ✓


In [ ]:
# ============================================
# EVALUATION METRICS  (paper Section 2.5)
# ============================================
from scipy import stats

def metrics_mse(y_true, y_pred):
    """Mean Squared Error — lower is better."""
    return float(np.mean((y_true - y_pred) ** 2))


def metrics_ci(y_true, y_pred):
    """
    Concordance Index — measures rank agreement.
    Range [0.5, 1.0]; 1.0 = perfect. Paper Eq. (4).
    """
    n = 0
    concordant = 0.0
    for i in range(len(y_true)):
        for j in range(i + 1, len(y_true)):
            if y_true[i] != y_true[j]:
                n += 1
                diff_true = y_true[i] - y_true[j]
                diff_pred = y_pred[i] - y_pred[j]
                if diff_true * diff_pred > 0:
                    concordant += 1.0
                elif diff_pred == 0:
                    concordant += 0.5
    return concordant / n if n > 0 else 0.0


def metrics_pearson(y_true, y_pred):
    """Pearson Correlation Coefficient. Paper Eq. (5)."""
    r, _ = stats.pearsonr(y_true, y_pred)
    return float(r)


def metrics_r2m(y_true, y_pred):
    """
    Squared Correlation Coefficient r²m.
    Penalises deviation from the regression line through origin.
    Paper Eq. (6)-(7).
    """
    # r² with intercept  (Eq. 6)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2     = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0

    # r0² without intercept — regression forced through origin
    ss_pred0 = np.sum(y_pred ** 2)
    ss_cross  = np.sum(y_true * y_pred)
    k         = ss_cross / ss_pred0 if ss_pred0 > 0 else 1.0
    ss_res0   = np.sum((y_true - k * y_pred) ** 2)
    r02        = 1.0 - ss_res0 / ss_tot if ss_tot > 0 else 0.0

    r2m = r2 * (1.0 - np.sqrt(abs(r2 - r02)))
    return float(r2m)


def evaluate_loader(model, loader, device, dataset_name="davis"):
    """
    Run inference on a DataLoader, return all four metrics.
    CI computation is O(n²) — fast enough for test sets ≤20K.
    """
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            preds = model(
                batch['smiles'].to(device),
                batch['drug_atomic'].to(device),
                batch['target_feat'].to(device),
                batch['target_bpe'].to(device),
                batch['residue'].to(device),
            )
            all_preds.append(preds.cpu().numpy())
            all_labels.append(batch['label'].numpy())

    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)

    return {
        'mse':     metrics_mse(y_true, y_pred),
        'ci':      metrics_ci(y_true, y_pred),
        'pearson': metrics_pearson(y_true, y_pred),
        'r2m':     metrics_r2m(y_true, y_pred),
    }


# ============================================
# TRAINING LOOP — ONE FOLD
# ============================================

def train_one_fold(
    fold_idx,
    train_loader,
    val_loader,
    test_loader,
    device,
    dataset_name,
    num_epochs   = 200,
    save_dir     = "/kaggle/working/checkpoints",
    smiles_vocab = SMILES_VOCAB_SIZE,
    bpe_vocab    = BPE_VOCAB_SIZE,
):
    """
    Train EDTA for one fold.

    Returns:
        best_val_metrics  : dict  — metrics at best val MSE epoch
        test_metrics      : dict  — metrics on test set using best weights
        train_losses      : list  — MSE per epoch (training)
        val_losses        : list  — MSE per epoch (validation)
    """
    os.makedirs(save_dir, exist_ok=True)
    ckpt_path = os.path.join(
        save_dir, f"{dataset_name}_fold{fold_idx+1}_best.pt"
    )

    # Fresh model each fold
    model = EDTA(
        smiles_vocab_size = smiles_vocab,
        bpe_vocab_size    = bpe_vocab,
    ).to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0002)

    best_val_mse     = float('inf')
    best_val_metrics = {}
    train_losses     = []
    val_losses       = []

    print(f"\n{'='*60}")
    print(f"  {dataset_name.upper()}  |  Fold {fold_idx+1}/5")
    print(f"{'='*60}")
    print(f"  {'Epoch':>6}  {'Train MSE':>10}  {'Val MSE':>10}  {'Val CI':>8}  {'Val r2m':>8}")
    print(f"  {'-'*6}  {'-'*10}  {'-'*10}  {'-'*8}  {'-'*8}")

    for epoch in range(1, num_epochs + 1):

        # ── Train ────────────────────────────────────────────────────
        model.train()
        epoch_loss = 0.0
        n_batches  = 0

        for batch in train_loader:
            optimizer.zero_grad()

            preds = model(
                batch['smiles'].to(device),
                batch['drug_atomic'].to(device),
                batch['target_feat'].to(device),
                batch['target_bpe'].to(device),
                batch['residue'].to(device),
            )
            loss = criterion(preds, batch['label'].to(device))
            loss.backward()

            # Gradient clipping — stabilises BiLSTM training
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

            optimizer.step()
            epoch_loss += loss.item()
            n_batches  += 1

        train_mse = epoch_loss / n_batches
        train_losses.append(train_mse)

        # ── Validate ─────────────────────────────────────────────────
        model.eval()
        val_loss  = 0.0
        n_val     = 0
        val_preds, val_labels = [], []

        with torch.no_grad():
            for batch in val_loader:
                preds = model(
                    batch['smiles'].to(device),
                    batch['drug_atomic'].to(device),
                    batch['target_feat'].to(device),
                    batch['target_bpe'].to(device),
                    batch['residue'].to(device),
                )
                loss = criterion(preds, batch['label'].to(device))
                val_loss   += loss.item()
                n_val      += 1
                val_preds.append(preds.cpu().numpy())
                val_labels.append(batch['label'].numpy())

        val_mse   = val_loss / n_val
        val_losses.append(val_mse)

        vp = np.concatenate(val_preds)
        vl = np.concatenate(val_labels)

        val_ci  = metrics_ci(vl, vp)
        val_r2m = metrics_r2m(vl, vp)

        # ── Save best ────────────────────────────────────────────────
        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_val_metrics = {
                'epoch':   epoch,
                'mse':     float(metrics_mse(vl, vp)),
                'ci':      float(val_ci),
                'pearson': float(metrics_pearson(vl, vp)),
                'r2m':     float(val_r2m),
            }
            torch.save(model.state_dict(), ckpt_path)

        # ── Log every 10 epochs ──────────────────────────────────────
        if epoch % 10 == 0 or epoch == 1:
            marker = " ◀ best" if val_mse == best_val_mse else ""
            print(
                f"  {epoch:>6}  {train_mse:>10.4f}  "
                f"{val_mse:>10.4f}  {val_ci:>8.4f}  "
                f"{val_r2m:>8.4f}{marker}"
            )

    # ── Evaluate test set with best weights ──────────────────────────
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    test_metrics = evaluate_loader(model, test_loader, device, dataset_name)

    print(f"\n  Best val epoch : {best_val_metrics['epoch']}")
    print(f"  Val  → MSE={best_val_metrics['mse']:.4f}  "
          f"CI={best_val_metrics['ci']:.4f}  "
          f"r={best_val_metrics['pearson']:.4f}  "
          f"r2m={best_val_metrics['r2m']:.4f}")
    print(f"  Test → MSE={test_metrics['mse']:.4f}  "
          f"CI={test_metrics['ci']:.4f}  "
          f"r={test_metrics['pearson']:.4f}  "
          f"r2m={test_metrics['r2m']:.4f}")

    return best_val_metrics, test_metrics, train_losses, val_losses


# ============================================
# FULL 5-FOLD CROSS-VALIDATION LOOP
# ============================================

def run_cross_validation(
    dataset_name,       # 'davis' or 'kiba'
    fold_loaders,       # list of (train_loader, val_loader, test_loader)
    device,
    num_epochs = 200,
    save_dir   = "/kaggle/working/checkpoints",
):
    """
    Run 5-fold CV and report averaged test metrics.

    fold_loaders : list of 5 tuples, each (train_loader, val_loader, test_loader)
                   — same test_loader for all folds (fixed hold-out set)
    """
    all_val_metrics  = []
    all_test_metrics = []
    all_train_losses = []
    all_val_losses   = []

    for fold_idx, (train_loader, val_loader, test_loader) in enumerate(fold_loaders):
        val_m, test_m, tr_loss, vl_loss = train_one_fold(
            fold_idx     = fold_idx,
            train_loader = train_loader,
            val_loader   = val_loader,
            test_loader  = test_loader,
            device       = device,
            dataset_name = dataset_name,
            num_epochs   = num_epochs,
            save_dir     = save_dir,
        )
        all_val_metrics.append(val_m)
        all_test_metrics.append(test_m)
        all_train_losses.append(tr_loss)
        all_val_losses.append(vl_loss)

    # ── Summary table (paper Table 5 format) ─────────────────────────
    print(f"\n{'='*60}")
    print(f"  {dataset_name.upper()}  5-FOLD CROSS-VALIDATION SUMMARY")
    print(f"{'='*60}")
    print(f"  {'':8}  {'MSE':>8}  {'CI':>8}  {'Pearson':>8}  {'r2m':>8}")
    print(f"  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}  {'-'*8}")

    for i, m in enumerate(all_test_metrics):
        print(
            f"  Fold {i+1:>3}  "
            f"{m['mse']:>8.4f}  {m['ci']:>8.4f}  "
            f"{m['pearson']:>8.4f}  {m['r2m']:>8.4f}"
        )

    # Average and std across folds
    for key in ['mse', 'ci', 'pearson', 'r2m']:
        vals = [m[key] for m in all_test_metrics]
        avg  = np.mean(vals)
        std  = np.std(vals)
        print(f"  {'Avg' if key=='mse' else '':8}  ", end="") if key=='mse' else None

    metrics_keys = ['mse', 'ci', 'pearson', 'r2m']
    avgs = {k: np.mean([m[k] for m in all_test_metrics]) for k in metrics_keys}
    stds = {k: np.std( [m[k] for m in all_test_metrics]) for k in metrics_keys}

    print(
        f"\n  {'Average':8}  "
        f"{avgs['mse']:>8.4f}  {avgs['ci']:>8.4f}  "
        f"{avgs['pearson']:>8.4f}  {avgs['r2m']:>8.4f}"
    )
    print(
        f"  {'Std Dev':8}  "
        f"{stds['mse']:>8.4f}  {stds['ci']:>8.4f}  "
        f"{stds['pearson']:>8.4f}  {stds['r2m']:>8.4f}"
    )

    return {
        'val_metrics':   all_val_metrics,
        'test_metrics':  all_test_metrics,
        'train_losses':  all_train_losses,
        'val_losses':    all_val_losses,
        'averages':      avgs,
        'std_devs':      stds,
    }


# ============================================
# RUN — DAVIS THEN KIBA
# ============================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if device.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# Build fold_loaders list expected by run_cross_validation
# davis_loaders[i] = (train_loader, val_loader, test_loader)
# test_loader is the same for all folds — fixed hold-out

davis_fold_loaders = [
    (train_l, val_l, davis_test_loader)
    for train_l, val_l, _ in davis_loaders
]

kiba_fold_loaders = [
    (train_l, val_l, kiba_test_loader)
    for train_l, val_l, _ in kiba_loaders
]

# ── DAVIS ─────────────────────────────────────────────────────────────
davis_results = run_cross_validation(
    dataset_name = 'davis',
    fold_loaders = davis_fold_loaders,
    device       = device,
    num_epochs   = 200,
    save_dir     = "/kaggle/working/checkpoints",
)

# ── KIBA ──────────────────────────────────────────────────────────────
kiba_results = run_cross_validation(
    dataset_name = 'kiba',
    fold_loaders = kiba_fold_loaders,
    device       = device,
    num_epochs   = 200,
    save_dir     = "/kaggle/working/checkpoints",
)

Device : cuda
GPU    : Tesla T4

  DAVIS  |  Fold 1/5
   Epoch   Train MSE     Val MSE    Val CI   Val r2m
  ------  ----------  ----------  --------  --------
       1     10.3150      0.6505    0.7305    0.1717 ◀ best
      10      0.5386      0.4395    0.8067    0.3747 ◀ best
      20      0.4221      0.4082    0.8370    0.3599
      30      0.3667      0.3622    0.8567    0.4229
      40      0.3348      0.3254    0.8574    0.4812
      50      0.3109      0.2848    0.8635    0.5820 ◀ best
      60      0.2790      0.3110    0.8639    0.4730
      70      0.2630      0.2703    0.8633    0.5977
      80      0.2372      0.2625    0.8651    0.6045 ◀ best
      90      0.2250      0.2541    0.8735    0.6725 ◀ best
     100      0.2105      0.2486    0.8769    0.6442 ◀ best
     110      0.2032      0.2451    0.8811    0.6409
     120      0.1881      0.2500    0.8774    0.6656
     130      0.1799      0.2667    0.8721    0.5532
     140      0.1767      0.2456    0.8699    0.6091
   